# SDH exp_003 — 전처리 10종 벤치마크

공용 `run_preprocessing_benchmark()`를 사용하여 모델과 5-Fold 조건은 고정하고 전처리만 비교합니다.

- 1차 실행: seed 42
- 평가 기준: OOF Macro F1
- 유망 후보만 seed 42/52/62로 반복 검증
- 빈도 필터와 hotspot은 각 fold의 train 데이터에서만 학습

In [ ]:
from pathlib import Path
import sys

search_paths = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path
        for path in search_paths
        if (path / "common").is_dir()
        and (path / "experiments").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_003_preprocessing.preprocessing import (
    make_preprocessing_candidates,
)
from experiments.SDH.exp_003_preprocessing.run_benchmark import run

print(f"프로젝트 루트: {PROJECT_ROOT}")

## 전처리 후보 확인

| Case | 전처리 |
| --- | --- |
| 01 | 결측→WT, WT/변이 이진화 |
| 02 | 01 + 상수 유전자 제거 |
| 03 | 02 + `log1p(변이 유전자 수)` |
| 04 | 02 + `log1p(실제 변이 토큰 수)` |
| 05 | 02 + 두 burden |
| 06 | 05 + 변이 유형별 개수 |
| 07 | 05 + 최소 변이 빈도 3 |
| 08 | 05 + 최소 변이 빈도 5 |
| 09 | 05 + 최소 변이 빈도 10 |
| 10 | 05 + 상위 50개 mutation-token hotspot |

In [ ]:
candidates = make_preprocessing_candidates()
list(candidates)

## 빠른 핵심 비교

먼저 baseline과 burden 관련 네 후보를 비교합니다. 각 후보마다 5-Fold 학습이 실행됩니다.

In [ ]:
core_cases = [
    "case_01_wt_binary",
    "case_03_gene_burden",
    "case_04_token_burden",
    "case_05_both_burdens",
]

core_leaderboard = run(selected_cases=core_cases)
core_leaderboard[
    [
        "preprocessing",
        "oof_f1_macro_mean",
        "oof_accuracy_mean",
        "fold_f1_macro_std",
        "elapsed_seconds",
    ]
]

## 10개 전체 비교

아래 셀은 10개 후보를 모두 실행하므로 시간이 오래 걸릴 수 있습니다.

In [ ]:
leaderboard = run()
leaderboard[
    [
        "preprocessing",
        "oof_f1_macro_mean",
        "oof_accuracy_mean",
        "fold_f1_macro_std",
        "elapsed_seconds",
    ]
]

## 유망 후보 반복 검증

1차 결과 상위 후보인 변이 유형, hotspot top 50, 최소 변이 빈도 10을 검증합니다. `confirmation=True`는 seed 42/52/62에서 각각 5-Fold를 실행하며 결과를 `_confirmation` 파일로 별도 저장합니다.

In [ ]:
best_cases = [
    "case_06_mutation_types",
    "case_10_hotspot_top50",
    "case_09_min_count_10",
]

confirmed_leaderboard = run(
    selected_cases=best_cases,
    confirmation=True,
)
confirmed_leaderboard[
    [
        "preprocessing",
        "oof_f1_macro_mean",
        "oof_f1_macro_std",
        "oof_accuracy_mean",
        "oof_accuracy_std",
    ]
]